In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# Đường dẫn file (thay nếu cần)
DATA_PATH = '/content/drive/MyDrive/PORTFOLIO/Delivery_Logistics.csv'

df_raw = pd.read_csv(DATA_PATH)

In [ ]:


print("Dataset shape:", df_raw.shape)

display(df_raw.head())

Dataset shape: (25000, 15)


,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost
0,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008,no,delivered,3,1632.7206
1,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,5,640.1700
2,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1448.1700
3,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008,no,delivered,3,1486.5700
4,250.99,dhl,clothing,van,two day,north,foggy,256.7,37.02,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016,no,delivered,4,1394.5600


In [ ]:
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df.columns.tolist())

['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition', 'distance_km', 'package_weight_kg', 'delivery_time_hours', 'expected_time_hours', 'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost']


**Kiểm tra schema**

In [ ]:
expected_columns = [
    "delivery_id",
    "delivery_partner",
    "package_type",
    "vehicle_type",
    "delivery_mode",
    "region",
    "weather_condition",
    "distance_km",
    "package_weight_kg",
    "delivery_time_hours",
    "expected_time_hours",
    "delayed",
    "delivery_status",
    "delivery_rating",
    "delivery_cost"
]

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

unexpected_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("Missing expected columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

Missing expected columns: []
Unexpected columns: []


**Data profiling**

In [ ]:
data_profile = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "missing_count": df.isnull().sum().values,
    "missing_percentage": (
        df.isnull().mean() * 100
    ).round(2).values,
    "unique_values": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ]
})

display(data_profile)

,column,data_type,missing_count,missing_percentage,unique_values
0,delivery_id,float64,0,0.0,24502
1,delivery_partner,object,0,0.0,9
2,package_type,object,0,0.0,9
3,vehicle_type,object,0,0.0,6
4,delivery_mode,object,0,0.0,4
5,region,object,0,0.0,5
6,weather_condition,object,0,0.0,6
7,distance_km,float64,0,0.0,2935
8,package_weight_kg,float64,0,0.0,4853
9,delivery_time_hours,object,0,0.0,20


**Missing value analysis**

In [ ]:
missing_summary = (
    df.isnull()
    .sum()
    .reset_index()
)

missing_summary.columns = [
    "column",
    "missing_count"
]

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(df) * 100
).round(2)

display(
    missing_summary
    .sort_values(
        "missing_percentage",
        ascending=False
    )
)

,column,missing_count,missing_percentage
0,delivery_id,0,0.0
1,delivery_partner,0,0.0
2,package_type,0,0.0
3,vehicle_type,0,0.0
4,delivery_mode,0,0.0
5,region,0,0.0
6,weather_condition,0,0.0
7,distance_km,0,0.0
8,package_weight_kg,0,0.0
9,delivery_time_hours,0,0.0


**Duplicate Delivery ID**

In [ ]:
duplicate_delivery_id = (
    df["delivery_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate delivery IDs:",
    duplicate_delivery_id
)

Duplicate delivery IDs: 498


In [ ]:
duplicate_records = df[
    df["delivery_id"].duplicated(
        keep=False
    )
].sort_values("delivery_id")

display(duplicate_records)

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost
0,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008,no,delivered,3,1632.7206
1,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,5,640.1700
2,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1448.1700
3,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008,no,delivered,3,1486.5700
20,250.99,fedex,automobile parts,scooter,two day,east,foggy,185.9,19.10,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000016,no,delivered,4,986.8000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24754,24750.01,fedex,documents,van,express,north,clear,67.2,37.45,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,4,498.3500
24753,24750.01,delhivery,electronics,van,same day,west,cold,267.4,45.05,1970-01-01 00:00:00.000000005,1970-01-01 00:00:00.000000008,no,delivered,5,1572.1500
24752,24750.01,blue dart,cosmetics,ev bike,express,west,clear,230.0,46.65,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000006,yes,delayed,3,1339.9500
24751,24750.01,blue dart,automobile parts,scooter,standard,north,clear,161.8,35.01,1970-01-01 00:00:00.000000004,1970-01-01 00:00:00.000000024,no,delivered,4,914.0300


**Data type conversion**

In [ ]:
numeric_columns = [
    "distance_km",
    "package_weight_kg",
    "delivery_time_hours",
    "expected_time_hours",
    "delivery_rating",
    "delivery_cost"
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [ ]:
df["delayed"].value_counts(dropna=False)

,count
delayed,
no,18331
yes,6669


In [ ]:
df["delayed"] = (
    df["delayed"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [ ]:
df["delayed_flag"] = np.where(
    df["delayed"] == "yes",
    1,
    np.where(
        df["delayed"] == "no",
        0,
        np.nan
    )
)

Kiểm tra categorical values

In [ ]:
categorical_columns = [
    "delivery_partner",
    "package_type",
    "vehicle_type",
    "delivery_mode",
    "region",
    "weather_condition",
    "delivery_status"
]

for col in categorical_columns:

    print("\n" + "=" * 60)
    print(col)

    display(
        df[col]
        .value_counts(dropna=False)
    )


delivery_partner


,count
delivery_partner,
xpressbees,2826
fedex,2818
dhl,2802
ekart,2801
blue dart,2798
delhivery,2786
shadowfax,2736
ecom express,2722
amazon logistics,2711



package_type


,count
package_type,
fragile items,2848
pharmacy,2810
documents,2805
automobile parts,2795
electronics,2792
clothing,2767
furniture,2746
cosmetics,2744
groceries,2693



vehicle_type


...

,count
region,
west,5095
central,5060
south,4977
north,4949
east,4919



weather_condition


,count
weather_condition,
foggy,4219
stormy,4198
rainy,4171
cold,4158
hot,4130
clear,4124



delivery_status


,count
delivery_status,
delivered,18331
delayed,5341
failed,1328


Chuẩn hóa categorical

In [ ]:
for col in categorical_columns:

    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
    )

Kiểm tra giá trị numeric bất thường

In [ ]:
quality_checks = {
    "negative_distance":
        (df["distance_km"] < 0).sum(),

    "negative_weight":
        (df["package_weight_kg"] < 0).sum(),

    "negative_delivery_time":
        (df["delivery_time_hours"] < 0).sum(),

    "negative_expected_time":
        (df["expected_time_hours"] < 0).sum(),

    "negative_cost":
        (df["delivery_cost"] < 0).sum(),

    "invalid_rating":
        (
            (df["delivery_rating"] < 0)
            |
            (df["delivery_rating"] > 5)
        ).sum()
}

quality_checks_df = pd.DataFrame(
    quality_checks.items(),
    columns=["check", "invalid_count"]
)

display(quality_checks_df)

,check,invalid_count
0,negative_distance,0
1,negative_weight,0
2,negative_delivery_time,0
3,negative_expected_time,0
4,negative_cost,0
5,invalid_rating,0


Tạo Delay Duration

In [ ]:
print("\nDATA TYPES")
print(
    df_raw[
        [
            "delivery_time_hours",
            "expected_time_hours",
            "delayed"
        ]
    ].dtypes
)


DATA TYPES
delivery_time_hours    object
expected_time_hours    object
delayed                object
dtype: object


In [ ]:
print("=== delivery_time_hours ===")
print(df_raw["delivery_time_hours"].head(30).tolist())

print("\n=== expected_time_hours ===")
print(df_raw["expected_time_hours"].head(30).tolist())

print("\n=== delayed ===")
print(df_raw["delayed"].head(30).tolist())

=== delivery_time_hours ===
['1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000002', '1970-01-01 00:00:00.000000010', '1970-01-01 00:00:00.000000006', '1970-01-01 00:00:00.000000009', '1970-01-01 00:00:00.000000004', '1970-01-01 00:00:00.000000006', '1970-01-01 00:00:00.000000004', '1970-01-01 00:00:00.000000005', '1970-01-01 00:00:00.000000003', '1970-01-01 00:00:00.000000005', '1970-01-01 00:00:00.000000003', '1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000006', '1970-01-01 00:00:00.000000012', '1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000010', '1970-01-01 00:00:00.000000011', '1970-01-01 00:00:00.000000004', '1970-01-01 00:00:00.000000007', '1970-01-01 00:00:00.000000006', '1970-01-01 00:00:00.000000009', '1970-01-01 00:00:00.000000006', '1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000007', '1970-01-01 00:00:00.000000006', '1970-01-01 00:00:00.000000009', '1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000003', '1970-01-01 00

In [ ]:
print("\nUnique delivery_time_hours:")
print(df_raw["delivery_time_hours"].unique()[:30])

print("\nUnique expected_time_hours:")
print(df_raw["expected_time_hours"].unique()[:30])


Unique delivery_time_hours:
['1970-01-01 00:00:00.000000008' '1970-01-01 00:00:00.000000002'
 '1970-01-01 00:00:00.000000010' '1970-01-01 00:00:00.000000006'
 '1970-01-01 00:00:00.000000009' '1970-01-01 00:00:00.000000004'
 '1970-01-01 00:00:00.000000005' '1970-01-01 00:00:00.000000003'
 '1970-01-01 00:00:00.000000012' '1970-01-01 00:00:00.000000011'
 '1970-01-01 00:00:00.000000007' '1970-01-01 00:00:00.000000013'
 '1970-01-01 00:00:00.000000000' '1970-01-01 00:00:00.000000001'
 '1970-01-01 00:00:00.000000014' '1970-01-01 00:00:00.000000016'
 '1970-01-01 00:00:00.000000015' '1970-01-01 00:00:00.000000017'
 '1970-01-01 00:00:00.000000018' '1970-01-01 00:00:00.000000019']

Unique expected_time_hours:
['1970-01-01 00:00:00.000000008' '1970-01-01 00:00:00.000000003'
 '1970-01-01 00:00:00.000000016' '1970-01-01 00:00:00.000000002'
 '1970-01-01 00:00:00.000000006' '1970-01-01 00:00:00.000000024'
 '1970-01-01 00:00:00.000000007' '1970-01-01 00:00:00.000000004'
 '1970-01-01 00:00:00.000000005

In [ ]:
def extract_encoded_value(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    try:
        # Take the number after the final decimal point
        return float(value.split(".")[-1])
    except:
        return np.nan

In [ ]:
df["delivery_time_hours"] = (
    df_raw["delivery_time_hours"]
    .apply(extract_encoded_value)
)

df["expected_time_hours"] = (
    df_raw["expected_time_hours"]
    .apply(extract_encoded_value)
)

In [ ]:
df["delay_duration_hours"] = (
    df["delivery_time_hours"]
    -
    df["expected_time_hours"]
)
display(
    df[
        [
            "delivery_time_hours",
            "expected_time_hours",
            "delay_duration_hours",
            "delayed"
        ]
    ].head(20)
)

,delivery_time_hours,expected_time_hours,delay_duration_hours,delayed
0,8.0,8.0,0.0,no
1,2.0,3.0,-1.0,no
2,10.0,16.0,-6.0,no
3,6.0,8.0,-2.0,no
4,9.0,16.0,-7.0,no
5,4.0,2.0,2.0,yes
6,6.0,8.0,-2.0,no
7,4.0,8.0,-4.0,no
8,5.0,8.0,-3.0,no
9,3.0,8.0,-5.0,no


In [ ]:
df["calculated_delayed_flag"] = np.where(
    df["delay_duration_hours"] > 0,
    1,
    np.where(
        df["delay_duration_hours"].notna(),
        0,
        np.nan
    )
)

In [ ]:
df["delay_reconciliation"] = np.where(
    (
        df["delayed_flag"].notna()
        &
        df["calculated_delayed_flag"].notna()
    ),
    np.where(
        df["delayed_flag"]
        ==
        df["calculated_delayed_flag"],
        "Match",
        "Mismatch"
    ),
    "Not Validated"
)

In [ ]:
display(
    df["delay_reconciliation"]
    .value_counts(dropna=False)
)

,count
delay_reconciliation,
Match,23797
Mismatch,1203


In [ ]:
distance_bins = [
    -np.inf,
    10,
    25,
    50,
    np.inf
]

distance_labels = [
    "0-10 km",
    "10-25 km",
    "25-50 km",
    "50+ km"
]

df["distance_group"] = pd.cut(
    df["distance_km"],
    bins=distance_bins,
    labels=distance_labels,
    right=False
)

In [ ]:
try:
    df["weight_group"] = pd.qcut(
        df["package_weight_kg"],
        q=3,
        labels=[
            "Light",
            "Medium",
            "Heavy"
        ],
        duplicates="drop"
    )
except ValueError:
    df["weight_group"] = pd.NA

In [ ]:
df["cost_per_km"] = np.where(
    (
        df["distance_km"].notna()
        &
        (df["distance_km"] > 0)
    ),
    df["delivery_cost"]
    /
    df["distance_km"],
    np.nan
)

In [ ]:
final_quality_report = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean() * 100
    ).round(2).values,
    "unique_values": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ]
})

display(final_quality_report)

,column,data_type,missing_count,missing_percentage,unique_values
0,delivery_id,float64,0,0.0,24502
1,delivery_partner,string,0,0.0,9
2,package_type,string,0,0.0,9
3,vehicle_type,string,0,0.0,6
4,delivery_mode,string,0,0.0,4
5,region,string,0,0.0,5
6,weather_condition,string,0,0.0,6
7,distance_km,float64,0,0.0,2935
8,package_weight_kg,float64,0,0.0,4853
9,delivery_time_hours,float64,0,0.0,20


In [ ]:
output_file = "delivery_logistics_clean.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Clean dataset exported successfully: {output_file}"
)

Clean dataset exported successfully: delivery_logistics_clean.csv


In [ ]:
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.head(10)

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost,delayed_flag,delay_duration_hours,calculated_delayed_flag,delay_reconciliation,distance_group,weight_group,cost_per_km
0,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,8.0,8.0,no,delivered,3,1632.7206,0.0,0.0,0.0,Match,50+ km,Heavy,5.497376
1,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,2.0,3.0,no,delivered,5,640.1700,0.0,-1.0,0.0,Match,50+ km,Heavy,7.144754
2,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,10.0,16.0,no,delivered,4,1448.1700,0.0,-6.0,0.0,Match,50+ km,Medium,5.294954
3,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,6.0,8.0,no,delivered,3,1486.5700,0.0,-2.0,0.0,Match,50+ km,Light,5.511939
4,250.99,dhl,clothing,van,two day,north,foggy,256.7,37.02,9.0,16.0,no,delivered,4,1394.5600,0.0,-7.0,0.0,Match,50+ km,Heavy,5.432645
5,250.99,amazon logistics,documents,ev bike,express,west,rainy,48.4,33.15,4.0,2.0,yes,delayed,3,391.4500,1.0,2.0,1.0,Match,25-50 km,Medium,8.087810
6,250.99,delhivery,groceries,scooter,same day,central,clear,198.3,43.79,6.0,8.0,no,delivered,3,1222.8700,0.0,-2.0,0.0,Match,50+ km,Heavy,6.166768
7,250.99,xpressbees,fragile items,van,same day,north,cold,114.6,42.63,4.0,8.0,no,delivered,3,800.8900,0.0,-4.0,0.0,Match,50+ km,Heavy,6.988569
8,250.99,blue dart,clothing,van,same day,south,hot,142.4,14.06,5.0,8.0,no,delivered,5,854.1800,0.0,-3.0,0.0,Match,50+ km,Light,5.998455
9,250.99,delhivery,pharmacy,truck,same day,east,foggy,47.1,29.28,3.0,8.0,no,delivered,5,423.3400,0.0,-5.0,0.0,Match,25-50 km,Medium,8.988110
